# 03-1. 確率の基礎 — 動かして確かめる

📖 解説: [`../01_probability_basics.md`](../01_probability_basics.md)

## このノートで触るもの
1. 大数の法則 (試行回数 vs 頻度) を可視化
2. 同時確率の確認
3. 条件付き確率
4. 独立 vs 排反
5. 包除原理

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (01_probability_basics.md)](../01_probability_basics.md)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, IntSlider

%matplotlib inline
rng = np.random.default_rng(seed=42)

## 1. 大数の法則を観察

In [ ]:
for n in [10, 100, 1_000, 100_000, 10_000_000]:
    flips = rng.integers(0, 2, size=n)
    print(f'n = {n:>10}  →  表の割合: {flips.mean():.6f}')
print('\n→ n を増やすほど 0.5 に近づく')

In [ ]:
# 試行回数の累積平均を可視化
n = 10_000
flips = rng.integers(0, 2, size=n)
cumulative = np.cumsum(flips) / np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cumulative, linewidth=1)
ax.axhline(0.5, color='red', linestyle='--', label='理論値 0.5')
ax.set_xlabel('試行回数')
ax.set_ylabel('表の累積割合')
ax.set_xscale('log')
ax.set_title('大数の法則: n を増やすほど 0.5 に収束')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 2. 同時確率と独立性 — 2 つのコイン

In [ ]:
n = 1_000_000
coin1 = rng.integers(0, 2, size=n)
coin2 = rng.integers(0, 2, size=n)

p_a = coin1.mean()
p_b = coin2.mean()
p_ab = ((coin1 == 1) & (coin2 == 1)).mean()

print(f'P(coin1 = 表)        = {p_a:.4f}')
print(f'P(coin2 = 表)        = {p_b:.4f}')
print(f'P(両方表) 実測       = {p_ab:.4f}')
print(f'P(coin1) × P(coin2) = {p_a * p_b:.4f}  ← 独立なら一致するはず')

## 3. 条件付き確率 — サイコロ 2 個

In [ ]:
n = 1_000_000
d1 = rng.integers(1, 7, size=n)
d2 = rng.integers(1, 7, size=n)
total = d1 + d2

# P(合計 = 7)
p_total7 = (total == 7).mean()
print(f'P(合計 = 7) = {p_total7:.4f}  (理論 6/36 = {6/36:.4f})')

# P(合計 = 7 | d1 = 3)
mask = (d1 == 3)
p_cond = (total[mask] == 7).mean()
print(f'P(合計 = 7 | d1 = 3) = {p_cond:.4f}  (理論 1/6 = {1/6:.4f})')

# 条件付き確率の方向違いの注意: P(d1=3 | 合計=7)
mask2 = (total == 7)
p_rev = (d1[mask2] == 3).mean()
print(f'P(d1 = 3 | 合計 = 7) = {p_rev:.4f}  (理論 1/6)')
print('\n→ どちらも 1/6 になるのはたまたま。一般には P(A|B) ≠ P(B|A)')

## 4. 独立 vs 排反 (混同しやすい)

In [ ]:
# 排反の例: 1 個のサイコロで「1 が出る」と「2 が出る」
die = rng.integers(1, 7, size=n)
p_1 = (die == 1).mean()
p_2 = (die == 2).mean()
p_12 = ((die == 1) & (die == 2)).mean()

print('--- 排反な例: 1 個のサイコロで P(1) と P(2) ---')
print(f'P(1)        = {p_1:.4f}')
print(f'P(2)        = {p_2:.4f}')
print(f'P(1 ∩ 2)    = {p_12:.4f}  ← 0 (同時には起きない = 排反)')
print(f'P(1) × P(2) = {p_1 * p_2:.4f}  ← 0 ではない')
print('→ 排反だが、独立ではない (一方が起きると他方は絶対起きない = 影響しまくり)')

## 5. 包除原理

In [ ]:
# トランプ: P(♠ ∪ Ace)
# A: ♠が出る, B: Aceが出る
# P(A) = 13/52, P(B) = 4/52, P(A∩B) = 1/52 (♠のAceだけ)

n = 1_000_000
# シミュレーション: 0~51 を引く. 0~12 = ♠, 13~25 = ♥, 26~38 = ♦, 39~51 = ♣
# 0,13,26,39 が Ace
cards = rng.integers(0, 52, size=n)

is_spade = cards < 13
is_ace = (cards % 13) == 0

p_spade = is_spade.mean()
p_ace = is_ace.mean()
p_both = (is_spade & is_ace).mean()
p_either = (is_spade | is_ace).mean()

print(f'P(♠)         = {p_spade:.4f}  (理論 13/52 = {13/52:.4f})')
print(f'P(Ace)       = {p_ace:.4f}  (理論 4/52 = {4/52:.4f})')
print(f'P(♠ ∩ Ace)   = {p_both:.4f}  (理論 1/52 = {1/52:.4f})')
print(f'P(♠ ∪ Ace)   = {p_either:.4f}')
print(f'包除原理: P(♠) + P(Ace) - P(♠∩Ace) = {p_spade + p_ace - p_both:.4f}  ← 一致!')

## 6. スライダー: 試行回数を変えて頻度の安定性を体感

In [ ]:
def show_convergence(n_pow: int) -> None:
    n = 10 ** n_pow
    rng_local = np.random.default_rng(42)
    flips = rng_local.integers(0, 2, size=n)
    cumulative = np.cumsum(flips) / np.arange(1, n + 1)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(cumulative, linewidth=1)
    ax.axhline(0.5, color='red', linestyle='--', label='理論 0.5')
    ax.set_xlabel('試行回数')
    ax.set_ylabel('表の累積割合')
    ax.set_title(f'n = 10^{n_pow} = {n:,} 回  最終値 = {cumulative[-1]:.6f}')
    ax.set_ylim(0.3, 0.7)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

interact(show_convergence, n_pow=IntSlider(min=1, max=6, value=3, description='log10(n)'));

## まとめ
- 大数の法則を実体験 (n→∞ で頻度 → 確率)
- 独立性は `P(A∩B) = P(A)P(B)` で判定
- 条件付き確率は方向に注意 `P(A|B) ≠ P(B|A)`
- 排反 ≠ 独立 (混同に注意)
- 包除原理 `P(A∪B) = P(A) + P(B) - P(A∩B)`

## 次へ
→ [`02_distributions.ipynb`](02_distributions.ipynb)  解説 → [`../02_distributions.md`](../02_distributions.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [章 TOP](../README.md) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`02_distributions.ipynb`](02_distributions.ipynb) |